# Mixed Precision Training

| Difficulty | ██████░░░░ 6/10 |
| :--- | :--- |
| Prerequisites | 06.05 |
| Time | ~70 min |
| Colab GPU | Needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/06_Numerical_Methods/06_mixed_precision_training.ipynb)

---

## 🎯 Learning Objective

Learn loss scaling and master-weight strategies that make float16/bfloat16 training numerically safe.

---

## 📐 Theory

`04.08` introduced the static loss-scaling trick: multiply the loss by a fixed large constant
before backpropagating, so small gradients don't underflow in float16 (recall `06.01`), then
divide the resulting gradients back down before the optimizer step. This notebook covers what
a *real* mixed-precision training loop does differently: the scale factor isn't fixed, it's
**dynamically adjusted** throughout training, and weights are kept in a separate, higher-precision
**master copy**.

### Why a fixed scale isn't good enough

A fixed scale faces a dilemma: too small, and gradients still underflow to zero in float16
(defeating the purpose); too large, and gradients that were previously fine now *overflow* to
`inf` instead. Since gradient magnitudes shift substantially over the course of training (large
early on, often much smaller as the model converges, and occasionally spiking on an unusual
batch), no single fixed scale works well for an entire run.

### Dynamic loss scaling

**Dynamic loss scaling** treats the scale factor as a moving target, adjusted based on
observed behavior:

1. Start with a large scale (e.g. $2^{16}$).
2. After each backward pass, check whether any gradient is `inf` or `NaN` (an **overflow**).
3. If an overflow occurred: **skip this optimizer step entirely** (the gradients are unusable)
   and **halve the scale** — it was too aggressive for the gradients just seen.
4. If no overflow occurred for `growth_interval` consecutive steps: **multiply the scale by a
   growth factor** (e.g. double it) — training has been stable, so try being more aggressive
   to reduce underflow risk.

This turns loss scaling from a single fixed hyperparameter you'd have to hand-tune into a
self-correcting feedback loop: the scale naturally settles near the largest value that doesn't
cause frequent overflows, without ever needing to know that value in advance.

### Master weights

Even with dynamic scaling, some information genuinely can't survive a round-trip through
float16's precision (recall `06.01`'s machine epsilon discussion) — particularly small weight
updates accumulating on top of much larger existing weights, a scenario prone to the same
catastrophic-cancellation-adjacent precision loss covered in `06.02`. **Mixed precision
training** keeps a **master copy of the weights in float32**, used for the actual optimizer
update; the float16/bfloat16 copies used for the fast forward/backward computation are
refreshed from that float32 master after every step. Compute happens fast, in low precision;
the accumulated *state* (the weights themselves, which must remember many small updates over
time) stays in high precision.

### Putting it together

A full mixed-precision training step looks like: (1) cast weights to float16/bfloat16, (2)
forward and backward pass in low precision, scaling the loss up first, (3) check for
overflow — if found, skip the step and shrink the scale, (4) otherwise, unscale the gradients,
apply the optimizer step to the float32 master weights, and periodically grow the scale back up.
None of this changes *what* is being computed mathematically — it's pure engineering on top of
`04.01`-`04.09`'s optimization theory, exactly as `04.08` noted for its own simpler tricks.

---

In [ ]:
# Setup
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Tracking a dynamic loss scaler's scale factor across training steps shows it steadily growing
while gradients stay well-behaved, then dropping sharply the instant an overflow is detected,
before resuming its climb — a live feedback loop, not a fixed number.

In [ ]:
class DynamicLossScaler:
    def __init__(self, init_scale=2.0**16, growth_factor=2.0, backoff_factor=0.5, growth_interval=3):
        self.scale = init_scale
        self.growth_factor, self.backoff_factor, self.growth_interval = growth_factor, backoff_factor, growth_interval
        self.steps_since_overflow = 0

    def update(self, overflowed):
        if overflowed:
            self.scale *= self.backoff_factor
            self.steps_since_overflow = 0
        else:
            self.steps_since_overflow += 1
            if self.steps_since_overflow >= self.growth_interval:
                self.scale *= self.growth_factor
                self.steps_since_overflow = 0

torch.manual_seed(0)
model_viz = torch.nn.Linear(4, 1).double()
scaler = DynamicLossScaler(growth_interval=3)
scale_history, overflow_steps = [], []

for step in range(24):
    x = torch.randn(4, dtype=torch.float64) * (1000 if step in (10, 18) else 1)  # two induced spikes
    model_viz.zero_grad()
    loss = (model_viz(x) ** 2).sum() * 1e-6
    (loss * scaler.scale).backward()
    overflowed = any(torch.isinf(p.grad.half()).any() for p in model_viz.parameters())
    scaler.update(overflowed)
    scale_history.append(scaler.scale)
    if overflowed:
        overflow_steps.append(step)

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(scale_history, 'o-', color="#4C72B0", markersize=4)
for s in overflow_steps:
    ax.axvline(s, color="#C44E52", ls="--", alpha=0.7)
ax.set_xlabel("training step"); ax.set_ylabel("loss scale factor (log scale)")
ax.set_title("Dynamic loss scaling: grows steadily, then\nhalves sharply right at each detected overflow (dashed lines)")
plt.show()

print(f"Overflow detected at steps: {overflow_steps}")
print(f"Scale grew from {scale_history[0]:.0f} to {scale_history[-1]:.0f} over {len(scale_history)} steps,")
print(f"dipping at each overflow before resuming growth -- a self-correcting feedback loop, not a fixed number.")

## 🐍 Implementation from Scratch

We implement the full dynamic-loss-scaling training loop -- scale, check for overflow, skip
or step, unscale, periodically grow -- and confirm a master-weights setup keeps float32
precision on the accumulated weights while computation happens in bfloat16.

In [ ]:
class DynamicLossScaler:
    """From-scratch version of torch.cuda.amp.GradScaler's core logic: grow the scale when
    training is stable, shrink it immediately on overflow, and always skip the optimizer
    step for any batch whose gradients actually overflowed."""
    def __init__(self, init_scale=2.0**16, growth_factor=2.0, backoff_factor=0.5, growth_interval=3):
        self.scale = init_scale
        self.growth_factor, self.backoff_factor, self.growth_interval = growth_factor, backoff_factor, growth_interval
        self.steps_since_overflow = 0

    def check_overflow(self, model):
        return any(torch.isinf(p.grad).any() or torch.isnan(p.grad).any()
                   for p in model.parameters() if p.grad is not None)

    def update(self, overflowed):
        if overflowed:
            self.scale *= self.backoff_factor
            self.steps_since_overflow = 0
        else:
            self.steps_since_overflow += 1
            if self.steps_since_overflow >= self.growth_interval:
                self.scale *= self.growth_factor
                self.steps_since_overflow = 0

def mixed_precision_step(model_master, model_low_precision, x, y, scaler, opt):
    """One full mixed-precision training step: sync low-precision copy from the float32
    master, compute in low precision, scale/check/unscale, then update the MASTER weights."""
    with torch.no_grad():
        for p_low, p_master in zip(model_low_precision.parameters(), model_master.parameters()):
            p_low.copy_(p_master.to(p_low.dtype))

    for p in model_low_precision.parameters():
        if p.grad is not None:
            p.grad = None
    loss = torch.nn.functional.mse_loss(model_low_precision(x.to(torch.bfloat16)), y.to(torch.bfloat16))
    (loss.float() * scaler.scale).backward()

    overflowed = scaler.check_overflow(model_low_precision)
    if not overflowed:
        opt.zero_grad()
        for p_low, p_master in zip(model_low_precision.parameters(), model_master.parameters()):
            p_master.grad = (p_low.grad.float() / scaler.scale)  # unscale, cast up to float32
        opt.step()  # updates the MASTER (float32) weights, not the low-precision copy
    scaler.update(overflowed)
    return loss.item(), overflowed

torch.manual_seed(2)
model_master = torch.nn.Sequential(torch.nn.Linear(4, 8), torch.nn.Tanh(), torch.nn.Linear(8, 1))
model_low = torch.nn.Sequential(torch.nn.Linear(4, 8), torch.nn.Tanh(), torch.nn.Linear(8, 1)).to(torch.bfloat16)
opt = torch.optim.SGD(model_master.parameters(), lr=0.05)
scaler2 = DynamicLossScaler(growth_interval=5)

rng = np.random.default_rng(0)
X = torch.tensor(rng.uniform(-2, 2, size=(50, 4)), dtype=torch.float32)
y = torch.sin(X.sum(dim=1, keepdim=True))

losses = []
for step in range(60):
    loss_val, overflowed = mixed_precision_step(model_master, model_low, X, y, scaler2, opt)
    losses.append(loss_val)
print(f"Mixed-precision training loop: loss {losses[0]:.4f} -> {losses[-1]:.4f}")
print(f"Final scale factor: {scaler2.scale:.0f}")

# --- Master weights: why float32 accumulation matters ---
w_master_fp32 = torch.tensor([1000.0], dtype=torch.float32)
w_direct_bf16 = torch.tensor([1000.0], dtype=torch.bfloat16)
small_update, n_updates = 0.01, 500
for _ in range(n_updates):
    w_master_fp32 -= small_update
    w_direct_bf16 -= torch.tensor(small_update, dtype=torch.bfloat16)
true_value = 1000.0 - small_update * n_updates
print(f"\nAfter {n_updates} updates of -{small_update} starting from 1000.0 (true result: {true_value}):")
print(f"  float32 master weight:        {w_master_fp32.item():.4f}")
print(f"  direct bfloat16 accumulation: {w_direct_bf16.item():.4f}  <- every update was silently lost")

## 🤖 Why This Matters for AI

`torch.cuda.amp.GradScaler` (and its more modern `torch.amp.GradScaler` equivalent) implements
exactly the dynamic loss-scaling algorithm derived above, and every major LLM training codebase
(Megatron-LM, DeepSpeed, HuggingFace's Trainer) either uses it directly or reimplements the same
grow/backoff logic. The reason it's ubiquitous rather than optional: without it, engineers would
otherwise need to hand-tune a fixed scale for every new model size, batch size, and learning
rate, and that scale would likely need re-tuning mid-training as gradient statistics shift. Below
we confirm the pieces above compose into a stable training loop, and time-travel back to
`06.01`'s bfloat16-vs-float32 precision discussion to show concretely why master weights in
float32 matter for anything that accumulates many small updates over a long training run.

In [ ]:
import torch.amp as amp

# Confirm PyTorch's own GradScaler exists and follows the SAME grow/backoff pattern as ours
# (enabled=False since this machine is CPU-only; on a CUDA GPU this would actively scale losses)
official_scaler = amp.GradScaler('cuda', enabled=False)
print(f"torch.amp.GradScaler starting scale: {official_scaler.get_scale()}")
print("On a CUDA GPU, GradScaler.scale(loss) / step(opt) / update() implements exactly the")
print("grow-on-success, backoff-on-overflow logic our DynamicLossScaler class reproduces above.")

# Re-run our full mixed-precision loop with a LARGER, longer-running model to make the
# master-weights precision benefit unambiguous over a longer horizon
torch.manual_seed(3)
model_master2 = torch.nn.Sequential(torch.nn.Linear(4, 16), torch.nn.Tanh(), torch.nn.Linear(16, 1))
model_low2 = torch.nn.Sequential(torch.nn.Linear(4, 16), torch.nn.Tanh(), torch.nn.Linear(16, 1)).to(torch.bfloat16)
opt2 = torch.optim.SGD(model_master2.parameters(), lr=0.05)
scaler3 = DynamicLossScaler(growth_interval=5)

losses2 = []
for step in range(150):
    loss_val, _ = mixed_precision_step(model_master2, model_low2, X, y, scaler3, opt2)
    losses2.append(loss_val)
print(f"\n150-step mixed-precision run: loss {losses2[0]:.4f} -> {losses2[-1]:.4f}")
print(f"final scale factor reached: {scaler3.scale:.0f}")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(losses2, color="#4C72B0", lw=2)
ax.set_xlabel("training step"); ax.set_ylabel("loss")
ax.set_title("Full mixed-precision training loop:\nbfloat16 compute, float32 master weights, dynamic scaling")
plt.show()

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. Trace through `DynamicLossScaler` by hand for 6 steps starting at `scale=1024`,
   `growth_factor=2`, `backoff_factor=0.5`, `growth_interval=2`, with overflows occurring at
   steps 3 and 5 (0-indexed). Write out the scale after each step, then verify with the class.

<details>
<summary>💡 Solution</summary>

Step 0 (no overflow, count=1): scale stays $1024$. Step 1 (no overflow, count=2, hits
`growth_interval`): scale doubles to $2048$, count resets. Step 2 (no overflow, count=1): scale
stays $2048$. Step 3 (overflow): scale halves to $1024$, count resets. Step 4 (no overflow,
count=1): scale stays $1024$. Step 5 (overflow): scale halves to $512$. Running the actual class
with these overflow steps gives the identical sequence
$(1024, 2048, 2048, 1024, 1024, 512)$.

</details>

### 💭 UNDERSTAND
2. Change `growth_interval` in the Visual Intuition cell from 3 to 10 (mentally, or by
   re-running it). How does a longer growth interval change how quickly the scale recovers
   after an overflow, and what's the tradeoff against a short growth interval?

<details>
<summary>💡 Solution</summary>

A longer growth interval makes the scale recover *more slowly* after an overflow-triggered
halving — it takes 10 consecutive clean steps to double again instead of 3, so the scaler spends
more time running at a conservative (smaller) scale, which means more gradients closer to the
underflow boundary during that stretch. The tradeoff: a short growth interval recovers
aggressiveness quickly but risks oscillating — repeatedly growing the scale only to immediately
overflow and halve again, wasting the optimizer step each overflow forces you to skip; a long
growth interval wastes fewer steps to repeated overflow but leaves the scale unnecessarily
conservative (and gradients more underflow-prone) for longer after any single disruptive batch.

</details>

### ✏️ DERIVE
3. `mixed_precision_step` unscales gradients via `p_low.grad.float() / scaler.scale` before the
   optimizer step, so that the optimizer always sees gradients at their true, unscaled
   magnitude regardless of what scale is currently in use. Derive why this unscaling step is
   necessary for correctness: show that without it, the *effective learning rate* applied by
   SGD would depend on the current loss scale, which changes over the course of training for
   reasons unrelated to the learning rate schedule.

<details>
<summary>💡 Solution outline</summary>

SGD's update is $w \leftarrow w - \eta \cdot g$ for learning rate $\eta$ and gradient $g$.
Scaling the loss by $s$ before backprop scales every gradient by the same factor $s$ (since
gradients are linear in the loss via the chain rule): the raw backward pass produces $s\cdot g$,
not $g$. If this scaled gradient were fed directly into the optimizer without dividing by $s$
first, the update would become $w \leftarrow w - \eta \cdot (s\cdot g) = w - (\eta s)\cdot g$ —
mathematically identical to running SGD with learning rate $\eta s$ instead of $\eta$. Since $s$
is dynamically adjusted (halved on overflow, doubled after a stable streak) for numerical reasons
entirely unrelated to the optimization schedule, an un-unscaled update would make the *effective*
learning rate silently jump around by factors of 2 whenever the scaler adjusts — dividing by
`scaler.scale` before the optimizer step exactly cancels the $s$ factor, restoring the true
gradient $g$ so $\eta$ (whatever the actual learning-rate schedule says, recall `04.06`) is the
only thing controlling step size.

</details>

### 🐍 IMPLEMENT
4. `check_overflow` currently checks only the model's own parameter gradients. Extend
   `DynamicLossScaler` with a method that also accepts and checks an arbitrary list of extra
   tensors (e.g. gradients living outside `model.parameters()`, such as a separately-optimized
   embedding table) for `inf`/`NaN`, returning `True` if *any* tensor in either group overflowed.

<details>
<summary>✅ How to know you're right</summary>

Construct a test case where the model's own parameter gradients are all finite but one tensor
in your extra list contains an injected `inf` or `NaN` (e.g.
`torch.tensor([float('inf'), 1.0])`); your extended overflow check must return `True` for that
case, and `False` when every tensor in both groups is finite. This matters practically because a
real training loop's gradients often live in more places than `model.parameters()` alone (extra
loss terms, auxiliary heads) — missing any of them would let a real overflow silently corrupt an
optimizer step the scaler believes was safe.

</details>

### 🤖 APPLY
5. A researcher training a mixture-of-experts model reports that a handful of experts
   consistently show up in `check_overflow`'s `inf`/`NaN` detections early in training, while
   the rest of the model never does. Using this notebook's dynamic-scaling framework, explain
   why a *per-layer* (or per-expert) loss scale might behave better here than the single global
   scale used throughout this notebook, and what risk a single global scale specifically creates
   for the well-behaved experts when a few problem experts keep forcing the scale down.

<details>
<summary>✅ What you should observe</summary>

A single global scale must be small enough to avoid overflowing the *worst* expert's gradients,
even though every other expert could safely tolerate a much larger scale — so the badly-behaved
experts are effectively setting the underflow risk for everyone else's gradients too, exactly
the failure mode the dynamic scaler's backoff mechanism responds to. A per-layer/per-expert
scale (used by some large-scale MoE training frameworks) decouples this: each expert gets its
own scale that grows or shrinks based only on its own overflow history, letting well-behaved
experts run at a more aggressive (better-underflow-protected) scale than the single global
scheme would ever allow, while still protecting the problem experts locally. The key connection
to make: this is the same "one size doesn't fit all when sub-components behave very
differently" theme as `04.04`'s per-parameter adaptive learning rates (Adam), just applied to
loss scale instead of step size.

</details>

### 🚀 CHALLENGE
6. In the master-weights comparison, reduce `small_update` from `0.01` to `0.001` and increase
   `n_updates` from `500` to `5000` (same total intended change of $-5.0$ from a starting weight
   of $1000.0$). Does the bfloat16 accumulation's final error get worse, better, or stay the
   same relative to before? Explain your answer in terms of bfloat16's representable-value gap
   (recall `06.01`'s machine epsilon) near a weight magnitude of $1000$, compared to the size of
   each individual update.

<details>
<summary>🔍 What a strong answer covers</summary>

The bfloat16 error is *exactly the same* in both cases: $5.0$ (the bf16 weight ends at exactly
$1000.0$, having accumulated zero net change either way), while the float32 master correctly
tracks the true value ($994.995$ and $995.117$ respectively, both close to the true $995.0$ up
to normal float32 rounding). The reason is that bfloat16's representable-value gap near $1000$
is about $7.8$ (computed as `1000 * bfloat16_eps`, with `bfloat16_eps ≈ 0.0078`) — *larger* than
either update size ($0.01$ or $0.001$), so `1000.0 - update` rounds right back to `1000.0` every
single time in bfloat16, for exactly the same reason `06.02`'s $(10^{20}+1)-10^{20}$ example
loses its "+1." Making the update *smaller* doesn't fix this — it makes it *relatively* worse (a
smaller fraction of the true intended change survives, though here neither survives at all),
because bfloat16's resolution near a fixed weight magnitude doesn't depend on how small your
update is; it depends only on the weight's own magnitude. A strong answer identifies this as the
precise mechanism master weights fix: float32 has a far finer gap near the same magnitude
(machine epsilon $\approx 1.2\times10^{-7}$, so a gap near $1000$ of about $1.2\times10^{-4}$,
tens of thousands of times finer than bfloat16's), comfortably resolving updates of either size
tested here.

</details>

---

## 📚 Further Reading
- Micikevicius et al., ["Mixed Precision Training"](https://arxiv.org/abs/1710.03740)
- PyTorch documentation, [Automatic Mixed Precision](https://pytorch.org/docs/stable/amp.html)
- NVIDIA, [Training With Mixed Precision (developer guide)](https://docs.nvidia.com/deeplearning/performance/mixed-precision-training/index.html)

---

*Next notebook: [Numerical Issues in Transformers](07_numerical_issues_in_transformers.ipynb)*